In [2]:
import openmc
import numpy as np
import matplotlib.pyplot as plt

In [3]:
"""Creazione uranio """

uo2_central = openmc.Material(1, "uo2_central")

uo2_central.add_nuclide("U235", 0.025)
uo2_central.add_nuclide("U238", 0.975)
uo2_central.add_nuclide("O16", 2)
uo2_central.set_density("g/cm3", 10)        #Densità da riguardare
uo2_central.temperature = 430

uo2_internal = openmc.Material(2, "uo2_internal")

uo2_internal.add_nuclide("U235", 0.035)
uo2_internal.add_nuclide("U238", 0.965)
uo2_internal.add_nuclide("O16", 2)
uo2_internal.set_density("g/cm3", 10)       #Densità da riguardare
uo2_internal.temperature = 430

uo2_external = openmc.Material(3, "uo2_external")

uo2_external.add_nuclide("U235", 0.05)
uo2_external.add_nuclide("U238", 0.95)
uo2_external.add_nuclide("O16", 2)
uo2_external.set_density("g/cm3", 10)       #Densità da riguardare
uo2_external.temperature = 430

uo2_poisoned= openmc.Material(4, "uo2_poisoned")

uo2_poisoned.add_nuclide("U235", 0.025)
uo2_poisoned.add_nuclide("U238", 0.975)
uo2_poisoned.add_nuclide("O16", 5)
uo2_poisoned.add_element("Gd", 2)
uo2_poisoned.set_density("g/cm3", 10)       #Densità da riguardare
uo2_poisoned.temperature = 430


"""Creazione Cladding, gape e acqua"""

zirconium = openmc.Material(name = "zirconium")
zirconium.add_element("Zr", 1)
zirconium.set_density("g/cm3", 6.5)
zirconium.temperature = 300 

gap = openmc.Material(name = "gap")         #parametri modificabile (da chiedere)
gap.add_element("He", 1)
gap.set_density("g/cm3", 1e-6)
gap.temperature = 300

water = openmc.Material(name = "water")
water.add_element("H", 2)
water.add_nuclide("O16", 1)
water.set_density("g/cm3", 7.262)       #la densità non è 1 per la pressione
water.temperature = 300
water.add_s_alpha_beta("c_H_in_H2O")

"""creazione AIC"""
Argento = openmc.Material(name = "Argento")
Argento.add_element("Ag", 1)
Argento.set_density("g/cm3", 10.5) 
Argento.temperature = 350                   #DA CHIEDERE

Indio = openmc.Material(name = "Indio")
Indio.add_element("In", 1)
Indio.set_density("g/cm3", 7.31) 
Indio.temperature = 350

Cadmio = openmc.Material(name = "Cadmio")
Cadmio.add_element("Cd", 1)
Cadmio.set_density("g/cm3", 8.65) 
Cadmio.temperature = 350

AIC = openmc.Material.mix_materials([Argento, Indio, Cadmio], [0.8, 0.15, 0.05], "wo") 
"""
creazione acciaio  """                   #DA CHIEDERE/RIFARE
Ferro = openmc.Material(name = "Ferro")
Ferro.add_element("Fe", 1)
Ferro.set_density("g/cm3", 7.87)
Ferro.temperature = 350

Carbonio = openmc.Material(name = "Carbonio")
Carbonio.add_element("C", 1)
Carbonio.set_density("g/cm3", 2.26)
Carbonio.temperature = 350

steel = openmc.Material.mix_materials([Ferro, Carbonio], [0.98, 0.02], "wo")



materials = openmc.Materials([uo2_external, uo2_internal, uo2_central, uo2_poisoned, zirconium, gap, water, AIC, steel])
materials.export_to_xml()

In [4]:
fuel_radius = 0.4095
gap_thickness = 0.0085          #0.4180-0.4095
cladding_thickness = 0.4750-0.4180
pin_pitch = 1.2618
pin_height = 200
num_cells = 17
lattice_length = pin_pitch * num_cells # 21.5313 (assembly pitch) - 2*0.0095 (gap between fuel and clad)


envelope_radius = lattice_length/2 + 20 # raggio che circonda tutto il core

fuel_cyl = openmc.ZCylinder(r=fuel_radius)
gap_cyl = openmc.ZCylinder(r=fuel_radius + gap_thickness)
clad_cyl = openmc.ZCylinder(r=fuel_radius + gap_thickness + cladding_thickness)

pin_upper_surface = openmc.ZPlane(z0=pin_height/2, boundary_type='vacuum')
pin_lower_surface = openmc.ZPlane(z0=-pin_height/2, boundary_type='vacuum')

fuel_region = -fuel_cyl 
gap_region = +fuel_cyl & -gap_cyl
clad_region = +gap_cyl & -clad_cyl
water_region = +clad_cyl 

fuel_cell_internal = openmc.Cell(region=fuel_region, fill=uo2_internal)
fuel_cell_central = openmc.Cell(region=fuel_region, fill=uo2_central)
fuel_cell_external = openmc.Cell(region=fuel_region, fill=uo2_external)
fuel_cell_poisoned = openmc.Cell(region=fuel_region, fill=uo2_poisoned)




gap_cell = openmc.Cell(region=gap_region, fill=gap)
clad_cell = openmc.Cell(region=clad_region, fill=zirconium)
water_cell = openmc.Cell(region=water_region, fill=water)


pin_universe = openmc.Universe(cells=[fuel_cell_central, gap_cell, clad_cell, water_cell])


In [5]:

lattice_length = pin_pitch * num_cells

envelope_radius = lattice_length/2 + 20


fuel_cyl = openmc.ZCylinder(r=fuel_radius)
gap_cyl = openmc.ZCylinder(r=fuel_radius + gap_thickness)
clad_cyl = openmc.ZCylinder(r=fuel_radius + gap_thickness + cladding_thickness)

pin_upper_surface = openmc.ZPlane(z0=pin_height/2,boundary_type='vacuum')
pin_lower_surface = openmc.ZPlane(z0=-pin_height/2,boundary_type='vacuum')

fuel_region = -fuel_cyl 
gap_region = +fuel_cyl & -gap_cyl
clad_region = +gap_cyl & -clad_cyl
water_region = +clad_cyl 
fuel_cell_central = openmc.Cell(region=fuel_region, fill=uo2_central)
gap_cell = openmc.Cell(region=gap_region, fill=gap)
clad_cell = openmc.Cell(region=clad_region, fill=zirconium)
water_cell = openmc.Cell(region=water_region, fill=water)


pin_universe = openmc.Universe(cells=[fuel_cell_central, gap_cell, clad_cell, water_cell])





In [6]:
lattice = openmc.RectLattice() 

lattice.lower_left = (-lattice_length/2, -lattice_length/2)
lattice.pitch = (pin_pitch, pin_pitch)
lattice.outer = openmc.Universe(cells=[openmc.Cell(fill=water)])
lattice.universes = np.tile(pin_universe, (num_cells, num_cells))
# oppure
# lattice.universes = [[pin_universe]*num_cells]*num_cells

outer_cyl = openmc.ZCylinder(r=envelope_radius, boundary_type='vacuum')
# upper_plane = openmc.ZPlane(z0=pin_height/2, boundary_type='vacuum')
# lower_plane = openmc.ZPlane(z0=-pin_height/2, boundary_type='vacuum')   
envelope_region = -outer_cyl & -pin_upper_surface & +pin_lower_surface
envelope_cell = openmc.Cell(region=envelope_region, fill=lattice)
assembly_universe = openmc.Universe(cells=[envelope_cell])

geometry = openmc.Geometry(assembly_universe)
geometry.export_to_xml()

In [7]:
plot_xy = openmc.Plot()
plot_xy.filename = "plot_xy_pratic"
plot_xy.basis = 'xy'
plot_xy.origin = (0,0,0)
plot_xy.width = (2*envelope_radius, 2*envelope_radius)
plot_xy.pixels = (800, 800)
plot_xy.color_by = 'material'
plot_xy.colors = {uo2_central: "orange", gap: "purple", zirconium: "grey", water: "lightblue"}

plot_xz = openmc.Plot()
plot_xz.filename = "plot_xz_pratic"
plot_xz.basis = 'xz'
plot_xz.origin = (0,0,0)
plot_xz.width = (2*envelope_radius, pin_height)
xz_ratio = plot_xy.width[0] / plot_xz.width[0]
plot_xz.pixels = (int(800 * xz_ratio), 800)
plot_xz.color_by = 'material'
plot_xz.colors = {uo2_central   : "orange", gap: "purple", zirconium: "grey", water: "lightblue"}

plot_xz_offset = openmc.Plot()
plot_xz_offset.filename = "plot_xz_offset_pratic"
plot_xz_offset.basis = 'xz'
plot_xz_offset.origin = (0,pin_pitch/2,0)
plot_xz_offset.width = (2*envelope_radius, pin_height)
xz_ratio = plot_xy.width[0] / plot_xz_offset.width[0]
plot_xz_offset.pixels = (int(800 * xz_ratio), 800)
plot_xz_offset.color_by = 'material'
plot_xz_offset.colors = {uo2_central   : "orange", gap: "purple", zirconium: "grey", water: "lightblue"}


plots = openmc.Plots([plot_xy, plot_xz, plot_xz_offset])
plots.export_to_xml()
openmc.plot_geometry()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [8]:
ll,ur = geometry.bounding_box
point = openmc.stats.Box(lower_left=ll, upper_right=ur)
source = openmc.IndependentSource(space=point, constraints={'fissionable': True})

settings = openmc.Settings()
settings.source = source
settings.batches = 1000
settings.inactive = 100
settings.particles = 1000
settings.temperature = {"method": "interpolation"}

settings.export_to_xml()

In [9]:
openmc.config["cross_sections"] = "/root/OpenMC/openmc/prova/endfb71/endfb-vii.1-hdf5/cross_sections.xml"
openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################